In [3]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import csv
import os
import re
import time
import random
from urllib.parse import urljoin

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

# =====================
# Config
# =====================
START_YEAR = 2010
END_YEAR = 2025  # adjust if needed

BASE = "https://pubsonline.informs.org"
OUT_FILE = os.path.join(os.getcwd(), "ISR_Issues.csv")


def clean(s):
    if not s:
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()


def normalize_url(href):
    href = (href or "").strip()
    if not href:
        return ""
    if href.startswith("http://") or href.startswith("https://"):
        return href
    return urljoin(BASE, href)


def get_driver() -> webdriver.Chrome:
    opts = Options()

    # Use YOUR real Chrome profile
    # (from chrome://version -> Profile Path)
    # Profile Path: /Users/keerthisagi/Library/Application Support/Google/Chrome/Default
    # => user-data-dir = /Users/keerthisagi/Library/Application Support/Google/Chrome
    #    profile-directory = Default
    opts.add_argument("user-data-dir=/Users/keerthisagi/Library/Application Support/Google/Chrome")
    opts.add_argument("profile-directory=Default")

    # Keep it non-headless so you can see/solve any verification
    # opts.add_argument("--headless=new")

    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--window-size=1400,900")

    driver = webdriver.Chrome(options=opts)
    return driver


def get_soup_with_selenium(driver, url, wait_sec=4):
    driver.get(url)
    time.sleep(wait_sec)
    html = driver.page_source
    return BeautifulSoup(html, "html.parser")


def iter_issues(year_from=START_YEAR, year_to=END_YEAR):
    """
    Generate (issue_url, vol_issue_label, year) for ISR, assuming 4 issues/year.
    Observed mapping: 2010 -> Vol 21, 2011 -> Vol 22, ...
    """
    for year in range(year_from, year_to + 1):
        volume = year - 1989  # 2010 -> 21, 2011 -> 22, ...
        for issue in range(1, 5):
            url = f"{BASE}/toc/isre/{volume}/{issue}"
            label = f"Vol {volume} Issue {issue}"
            yield url, label, year


def extract_articles_from_issue(driver, issue_url, vol_issue_label, year):
    """From an issue TOC page, collect (Title, URL, Volume Issue, Vol Issue Year) rows."""
    try:
        soup = get_soup_with_selenium(driver, issue_url, wait_sec=4)
    except Exception as e:
        print(f"   Skipping issue (selenium error): {e}")
        return []

    rows = []
    seen_urls = set()

    # Try explicit title links first
    title_link_selectors = [
        "h5.issue-item_title a",
        "h5.issue-item__title a",
        "h4.issue-item_title a",
        "h4.issue-item__title a",
        "div.issue-item__title a",
        "div.issue-item_title a",
    ]

    for css in title_link_selectors:
        for a in soup.select(css):
            href = normalize_url(a.get("href"))
            title = clean(a.get_text())
            if not href or len(title) < 5:
                continue
            if href in seen_urls:
                continue
            seen_urls.add(href)
            rows.append([title, href, vol_issue_label, str(year)])

    if rows:
        return rows

    # Fallback: pair DOI links with nearest heading
    bad_href_parts = [
        "/doi/epdf/",
        "/doi/pdf/",
        "/doi/fpi/",
        "/doi/ref/",
        "/doi/references/",
        "rightslink",
        "/servlet/linkout",
    ]

    ignore_heading = {
        "abstract",
        "preview abstract",
        "research article",
        "editorial notes",
        "research commentary",
        "research note",
        "editorial",
        "erratum",
        "corrigendum",
    }

    ignore_link_text = {
        "permissions",
        "pdf",
        "full-text",
        "full text",
        "references",
        "first page",
        "abstract",
    }

    for a in soup.select("a[href]"):
        href = normalize_url(a.get("href") or "")
        if "10.1287" not in href:
            continue
        if any(p in href for p in bad_href_parts):
            continue
        if href in seen_urls:
            continue

        # nearest previous heading as title
        title = None
        h = a.find_previous(["h5", "h4", "h3"])
        while h is not None:
            ht = clean(h.get_text())
            htl = ht.lower()
            if ht and len(ht) >= 5 and htl not in ignore_heading:
                if not htl.startswith("pages") and "published online" not in htl:
                    title = ht
                    break
            h = h.find_previous(["h5", "h4", "h3"])

        if not title:
            lt = clean(a.get_text())
            if lt.lower() in ignore_link_text or len(lt) < 5:
                continue
            title = lt

        seen_urls.add(href)
        rows.append([title, href, vol_issue_label, str(year)])

    return rows


def write_rows(rows):
    file_exists = os.path.exists(OUT_FILE) and os.path.getsize(OUT_FILE) > 0

    with open(OUT_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Title", "URL", "Volume Issue", "Vol Issue Year"])
        writer.writerows(rows)


def main():
    # Start fresh each run
    if os.path.exists(OUT_FILE):
        os.remove(OUT_FILE)

    driver = get_driver()
    total_rows = 0

    try:
        all_issues = list(iter_issues(START_YEAR, END_YEAR))
        print(f"Total potential issues ({START_YEAR}–{END_YEAR}):", len(all_issues))

        for idx, (issue_url, vol_issue_label, year) in enumerate(all_issues, start=1):
            print(f"[{idx}/{len(all_issues)}] {vol_issue_label} | Year {year}")
            print("   URL:", issue_url)

            rows = extract_articles_from_issue(driver, issue_url, vol_issue_label, year)

            if rows:
                write_rows(rows)
                total_rows += len(rows)
                print(f"   Saved {len(rows)} rows")
            else:
                print("   No article rows found (or page inaccessible)")

            time.sleep(random.uniform(0.8, 1.6))

    finally:
        driver.quit()

    print("\nDone")
    print("Total rows saved:", total_rows)
    print("Output:", OUT_FILE)


if __name__ == "__main__":
    main()